In [1]:
import pandas as pd
import numpy as np
import requests
import time
import re
import json
from datetime import date
from dateutil.relativedelta import relativedelta

# --- SECURITY: use Kaggle Secrets instead of a hardcoded token ---
from kaggle_secrets import UserSecretsClient
api = UserSecretsClient().get_secret("CARBON_MAPPER_TOKEN")  # add this secret first: Add-ons -> Secrets

BASE_URL = "https://api.carbonmapper.org/api/v1"
HEADERS = {"Authorization": f"Bearer {api}"}
NM_PERMIAN_BBOX = [-104.6, 32.0, -103.0, 32.8]  # lon_min, lat_min, lon_max, lat_max (NM-only, TX excluded)

In [2]:
# ============================================================
# PHASE 1 — Load, clean, and cross-check plume + source files
# ============================================================
PLUMES_PATH = "/kaggle/input/datasets/miracleajoku/retrieved-dataset-geojson/plumes_2026-09-08T10_14_56.089Z.csv"
SOURCES_PATH = "/kaggle/input/datasets/miracleajoku/retrieved-dataset-geojson/sources_2026-09-08T10_14_53.468Z.json"

df_plumes = pd.read_csv(PLUMES_PATH)
df_plumes["datetime"] = pd.to_datetime(df_plumes["datetime"], utc=True, format="mixed")
before = len(df_plumes)
df_plumes = df_plumes[df_plumes["region"] == "New Mexico"].copy()
print(f"Dropped {before - len(df_plumes)} non-NM rows -> {len(df_plumes)} plumes remain")

with open(SOURCES_PATH) as f:
    sources_geojson = json.load(f)
source_records = []
for feat in sources_geojson["features"]:
    props = feat["properties"].copy()
    props["longitude"] = feat["geometry"]["coordinates"][0]
    props["latitude"] = feat["geometry"]["coordinates"][1]
    source_records.append(props)
df_sources = pd.DataFrame(source_records)
print(f"Sources loaded: {df_sources.shape}")

df_plumes.to_parquet("/kaggle/working/nm_plumes_clean.parquet")
df_sources.to_parquet("/kaggle/working/nm_sources_clean.parquet")

Dropped 19 non-NM rows -> 3944 plumes remain
Sources loaded: (1292, 18)


In [3]:
# ============================================================
# PHASE 3 — Assign persistence classes
# ============================================================
MIN_RELIABLE_OBS = 5
MIN_ULTRA_OBS = 10

def assign_persistence_class(row, min_reliable=MIN_RELIABLE_OBS, min_ultra=MIN_ULTRA_OBS):
    if row["observation_date_count"] < min_reliable:
        return "insufficient_evidence"
    if row["detection_date_count"] == 1:
        return "isolated"
    if row["persistence"] >= 0.5:
        return "ultra_persistent" if row["observation_date_count"] >= min_ultra else "persistent"
    if row["persistence"] >= 0.25:
        return "persistent"
    return "intermittent"

df_sources["persistence_class"] = df_sources.apply(assign_persistence_class, axis=1)

# Sensitivity variants (used later for class-stability / kappa reporting)
df_sources["persistence_class_sens_low"] = df_sources.apply(
    lambda r: assign_persistence_class(r, min_reliable=2, min_ultra=5), axis=1)
df_sources["persistence_class_sens_high"] = df_sources.apply(
    lambda r: assign_persistence_class(r, min_reliable=10, min_ultra=15), axis=1)
df_sources["persistence_class_sens_3"] = df_sources.apply(
    lambda r: assign_persistence_class(r, min_reliable=3, min_ultra=7), axis=1)
df_sources["persistence_class_sens_15"] = df_sources.apply(
    lambda r: assign_persistence_class(r, min_reliable=15, min_ultra=20), axis=1)

print(df_sources["persistence_class"].value_counts())
df_sources.to_parquet("/kaggle/working/nm_sources_labeled.parquet")

persistence_class
isolated                 610
intermittent             360
insufficient_evidence    172
persistent               131
ultra_persistent          19
Name: count, dtype: int64


In [4]:
# ============================================================
# PHASE 4 — Reconstruct daily sequences and build early-window features
# (final version: day-level collapse + correct per-day emission linkage)
# ============================================================
KNOWN_MISMATCH_IDS = [
    "CH4_1B2_250m_-104.11754_32.05183?status=not_deleted",
    "CH4_1B2_250m_-104.11947_32.02271?status=not_deleted",
]  # two sources with a known, negligible (0.15%) plume-ID edge case; excluded

df_sources_clean = df_sources[~df_sources["source_name"].isin(KNOWN_MISMATCH_IDS)].copy()
plume_emission_lookup = dict(zip(df_plumes["plume_id"], df_plumes["emission_auto"]))
source_plume_ids = dict(zip(df_sources_clean["source_name"], df_sources_clean["plume_ids"]))

SCENE_TS_RE = re.compile(r"(\d{8})t(\d{6})")
def parse_scene_timestamp(scene_name):
    m = SCENE_TS_RE.search(scene_name)
    if not m:
        return pd.NaT
    date_part, time_part = m.groups()
    try:
        return pd.to_datetime(f"{date_part}{time_part}", format="%Y%m%d%H%M%S", utc=True)
    except Exception:
        return pd.NaT

records = []
for _, row in df_sources_clean.iterrows():
    scene_names = row["observation_scenes_names"]
    plume_ids = list(row["plume_ids"]) if isinstance(row["plume_ids"], (list, np.ndarray)) else []
    for scene_entry in scene_names:
        scene_name = scene_entry.split(":")[-1]
        ts = parse_scene_timestamp(scene_name)
        if pd.isna(ts):
            continue
        matched_pid = next((pid for pid in plume_ids if scene_name in pid), None)
        emission_val = plume_emission_lookup.get(matched_pid, np.nan) if matched_pid else np.nan
        records.append({"source_id": row["source_name"], "timestamp": ts,
                         "is_detection": matched_pid is not None, "emission_auto": emission_val})

df_opp = pd.DataFrame(records)
df_opp["obs_date"] = df_opp["timestamp"].dt.date

# Collapse same-day multi-scene observations into one opportunity (validated: 99.8% agreement w/ CM's own counts)
df_daily = (df_opp.groupby(["source_id", "obs_date"], as_index=False)
            .agg(is_detection=("is_detection", "max"), timestamp=("timestamp", "min"),
                 emission_auto=("emission_auto", "max")))
df_daily = df_daily.sort_values(["source_id", "timestamp"]).reset_index(drop=True)
df_daily.to_parquet("/kaggle/working/nm_daily_full.parquet")

# Early-window (k=3,5,10) leak-proof features: use ONLY the first k rows per source
def build_features_for_window(k):
    rows = []
    for source_id, grp in df_daily.groupby("source_id"):
        grp = grp.sort_values("timestamp").reset_index(drop=True)
        if len(grp) < k:
            continue
        early = grp.iloc[:k]
        n_obs_early = len(early)
        n_detect_early = int(early["is_detection"].sum())
        detect_rate_early = n_detect_early / n_obs_early
        span_days_early = (early["timestamp"].max() - early["timestamp"].min()).total_seconds() / 86400
        early_detections = early[early["is_detection"]]
        first_gap_days = ((early_detections["timestamp"].min() - early["timestamp"].min()).total_seconds() / 86400
                           if len(early_detections) > 0 else np.nan)
        early_emissions = early_detections["emission_auto"].dropna()
        rows.append({
            "source_id": source_id, "window_k": k, "n_obs_early": n_obs_early,
            "n_detect_early": n_detect_early, "detect_rate_early": detect_rate_early,
            "span_days_early": span_days_early, "first_gap_days": first_gap_days,
            "first_emission_early": early_emissions.iloc[0] if len(early_emissions) > 0 else np.nan,
            "max_emission_early": early_emissions.max() if len(early_emissions) > 0 else np.nan,
            "mean_emission_early": early_emissions.mean() if len(early_emissions) > 0 else np.nan,
            "n_emission_readings_early": len(early_emissions),
        })
    return pd.DataFrame(rows)

label_cols = ["source_name", "persistence_class", "persistence", "detection_date_count", "observation_date_count"]
labels = df_sources_clean[label_cols].rename(columns={"source_name": "source_id"})

for k in [3, 5, 10]:
    feats = build_features_for_window(k)
    merged = feats.merge(labels, on="source_id", how="left")
    merged.to_parquet(f"/kaggle/working/nm_features_k{k}_v2.parquet")
    print(f"k={k}: {len(merged)} sources saved")

k=3: 1262 sources saved
k=5: 1118 sources saved
k=10: 964 sources saved


In [5]:
# ============================================================
# PHASE 5 / STEP 1 — Nested cross-validation for honest hyperparameter selection
# ============================================================
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
import xgboost as xgb
import itertools

df_k5 = pd.read_parquet("/kaggle/working/nm_features_k5_v2.parquet")
df_k5["target_persistent"] = df_k5["persistence_class"].isin(["persistent", "ultra_persistent"]).astype(int)

FEATURE_COLS = ["n_obs_early", "n_detect_early", "detect_rate_early", "span_days_early",
                "first_gap_days", "first_emission_early", "max_emission_early",
                "mean_emission_early", "n_emission_readings_early"]
X = df_k5[FEATURE_COLS].copy()
for col in ["first_emission_early", "max_emission_early", "mean_emission_early", "first_gap_days"]:
    X[f"{col}_missing"] = X[col].isna().astype(int)
    X[col] = X[col].fillna(X[col].median())
y = df_k5["target_persistent"].values

param_grid = {"max_depth": [2, 3, 4], "n_estimators": [100, 200, 300], "learning_rate": [0.03, 0.05, 0.1],
              "reg_lambda": [1.0, 3.0, 5.0], "min_child_weight": [3, 5, 8],
              "subsample": [0.7, 1.0], "colsample_bytree": [0.7, 1.0]}
keys, values = zip(*param_grid.items())
all_configs = [dict(zip(keys, v)) for v in itertools.product(*values)]

outer_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=1)
outer_scores, selected_configs = [], []

for train_idx, test_idx in outer_skf.split(X, y):
    X_tr_outer, X_te_outer = X.iloc[train_idx], X.iloc[test_idx]
    y_tr_outer, y_te_outer = y[train_idx], y[test_idx]
    best_cfg, best_auprc = None, -1
    for cfg in all_configs:
        inner_scores = []
        for ti, vi in inner_skf.split(X_tr_outer, y_tr_outer):
            pw = (y_tr_outer[ti] == 0).sum() / max((y_tr_outer[ti] == 1).sum(), 1)
            m = xgb.XGBClassifier(**cfg, scale_pos_weight=pw, tree_method="hist", eval_metric="aucpr", random_state=42)
            m.fit(X_tr_outer.iloc[ti], y_tr_outer[ti])
            inner_scores.append(average_precision_score(y_tr_outer[vi], m.predict_proba(X_tr_outer.iloc[vi])[:, 1]))
        if np.mean(inner_scores) > best_auprc:
            best_auprc, best_cfg = np.mean(inner_scores), cfg
    pw_outer = (y_tr_outer == 0).sum() / max((y_tr_outer == 1).sum(), 1)
    final_m = xgb.XGBClassifier(**best_cfg, scale_pos_weight=pw_outer, tree_method="hist", eval_metric="aucpr", random_state=42)
    final_m.fit(X_tr_outer, y_tr_outer)
    proba = final_m.predict_proba(X_te_outer)[:, 1]
    outer_scores.append({"auroc": roc_auc_score(y_te_outer, proba), "auprc": average_precision_score(y_te_outer, proba)})
    selected_configs.append(best_cfg)
    print(f"Outer fold: AUROC={outer_scores[-1]['auroc']:.3f}, AUPRC={outer_scores[-1]['auprc']:.3f}, config={best_cfg}")

print(f"\nNested CV: AUROC={np.mean([s['auroc'] for s in outer_scores]):.3f}, "
      f"AUPRC={np.mean([s['auprc'] for s in outer_scores]):.3f}")
# max_depth=4 and learning_rate=0.03 were the stable winners across folds -> locked into FINAL_CONFIG below

Outer fold: AUROC=0.773, AUPRC=0.531, config={'max_depth': 4, 'n_estimators': 100, 'learning_rate': 0.03, 'reg_lambda': 5.0, 'min_child_weight': 5, 'subsample': 1.0, 'colsample_bytree': 0.7}
Outer fold: AUROC=0.761, AUPRC=0.583, config={'max_depth': 4, 'n_estimators': 100, 'learning_rate': 0.03, 'reg_lambda': 3.0, 'min_child_weight': 8, 'subsample': 0.7, 'colsample_bytree': 1.0}
Outer fold: AUROC=0.804, AUPRC=0.654, config={'max_depth': 4, 'n_estimators': 100, 'learning_rate': 0.03, 'reg_lambda': 1.0, 'min_child_weight': 3, 'subsample': 0.7, 'colsample_bytree': 0.7}
Outer fold: AUROC=0.728, AUPRC=0.579, config={'max_depth': 4, 'n_estimators': 100, 'learning_rate': 0.03, 'reg_lambda': 5.0, 'min_child_weight': 5, 'subsample': 1.0, 'colsample_bytree': 0.7}
Outer fold: AUROC=0.738, AUPRC=0.538, config={'max_depth': 4, 'n_estimators': 300, 'learning_rate': 0.1, 'reg_lambda': 3.0, 'min_child_weight': 8, 'subsample': 1.0, 'colsample_bytree': 1.0}

Nested CV: AUROC=0.761, AUPRC=0.577


In [6]:
# ============================================================
# PHASE 5 / STEP 2 — Final production model, calibration, SHAP, ablation
# ============================================================
from sklearn.calibration import CalibratedClassifierCV
import shap

FINAL_CONFIG = {"max_depth": 4, "learning_rate": 0.03, "n_estimators": 200,
                 "reg_lambda": 5.0, "min_child_weight": 8, "subsample": 0.8, "colsample_bytree": 0.8}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
aurocs, auprcs, briers = [], [], []
from sklearn.metrics import brier_score_loss
for tr, te in skf.split(X, y):
    pw = (y[tr] == 0).sum() / max((y[tr] == 1).sum(), 1)
    cal = CalibratedClassifierCV(xgb.XGBClassifier(**FINAL_CONFIG, scale_pos_weight=pw, tree_method="hist",
                                                     eval_metric="aucpr", random_state=42), method="isotonic", cv=3)
    cal.fit(X.iloc[tr], y[tr])
    proba = cal.predict_proba(X.iloc[te])[:, 1]
    aurocs.append(roc_auc_score(y[te], proba)); auprcs.append(average_precision_score(y[te], proba))
    briers.append(brier_score_loss(y[te], proba))
print(f"Final model: AUROC={np.mean(aurocs):.3f}±{np.std(aurocs):.3f}, "
      f"AUPRC={np.mean(auprcs):.3f}±{np.std(auprcs):.3f}, Brier={np.mean(briers):.3f}±{np.std(briers):.3f}")

pw_full = (y == 0).sum() / (y == 1).sum()
final_model = xgb.XGBClassifier(**FINAL_CONFIG, scale_pos_weight=pw_full, tree_method="hist",
                                  eval_metric="aucpr", random_state=42)
final_model.fit(X, y)
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X)
print("\nSHAP importance:")
print(pd.Series(np.abs(shap_values).mean(axis=0), index=X.columns).sort_values(ascending=False))

# Ablation: drop emission-magnitude features
NON_EMISSION_COLS = [c for c in X.columns if "emission" not in c]
X_ablated = X[NON_EMISSION_COLS]
ablated_auprcs = []
for tr, te in skf.split(X_ablated, y):
    pw = (y[tr] == 0).sum() / max((y[tr] == 1).sum(), 1)
    m = xgb.XGBClassifier(**FINAL_CONFIG, scale_pos_weight=pw, tree_method="hist", eval_metric="aucpr", random_state=42)
    m.fit(X_ablated.iloc[tr], y[tr])
    ablated_auprcs.append(average_precision_score(y[te], m.predict_proba(X_ablated.iloc[te])[:, 1]))
print(f"\nAblation (no emission features) AUPRC: {np.mean(ablated_auprcs):.3f}±{np.std(ablated_auprcs):.3f}")

import joblib
final_model_calibrated = CalibratedClassifierCV(
    xgb.XGBClassifier(**FINAL_CONFIG, scale_pos_weight=pw_full, tree_method="hist", eval_metric="aucpr", random_state=42),
    method="isotonic", cv=3)
final_model_calibrated.fit(X, y)
joblib.dump(final_model_calibrated, "/kaggle/working/persistence_classifier_calibrated.pkl")
joblib.dump(list(X.columns), "/kaggle/working/persistence_classifier_feature_order.pkl")

Final model: AUROC=0.772±0.042, AUPRC=0.564±0.055, Brier=0.080±0.006

SHAP importance:
n_detect_early                  0.711507
span_days_early                 0.430507
detect_rate_early               0.126299
first_gap_days                  0.078698
mean_emission_early             0.071042
first_emission_early            0.062431
max_emission_early              0.051390
first_gap_days_missing          0.016927
n_emission_readings_early       0.002130
n_obs_early                     0.000000
first_emission_early_missing    0.000000
max_emission_early_missing      0.000000
mean_emission_early_missing     0.000000
dtype: float32

Ablation (no emission features) AUPRC: 0.575±0.043


['/kaggle/working/persistence_classifier_feature_order.pkl']

In [7]:
# ============================================================
# PHASE 6 — Survival dataset, then final confound-corrected Cox model
# ============================================================
!pip install lifelines --quiet
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.utils import concordance_index

df_daily = pd.read_parquet("/kaggle/working/nm_daily_full.parquet")
df_sources_full = pd.read_parquet("/kaggle/working/nm_sources_labeled.parquet")
GLOBAL_MAX_DATE = pd.to_datetime(df_daily["timestamp"], utc=True).max()
FOLLOWUP_CENSOR_DAYS = 365

records = []
for source_id, grp in df_daily.groupby("source_id"):
    grp = grp.sort_values("timestamp")
    detections = grp[grp["is_detection"]]
    if len(detections) == 0:
        continue
    first_detect, last_detect = detections["timestamp"].min(), detections["timestamp"].max()
    duration_days = max((last_detect - first_detect).total_seconds() / 86400, 0.01)
    days_since_last = (GLOBAL_MAX_DATE - last_detect).total_seconds() / 86400
    event_observed = 1 if days_since_last >= FOLLOWUP_CENSOR_DAYS else 0
    records.append({"source_id": source_id, "duration_days": duration_days, "event_observed": event_observed})
df_survival = pd.DataFrame(records)
df_survival.to_parquet("/kaggle/working/nm_survival_data.parquet")

df_k5_covs = pd.read_parquet("/kaggle/working/nm_features_k5_v2.parquet")[
    ["source_id", "n_detect_early", "detect_rate_early", "span_days_early", "first_gap_days"]].copy()
df_k5_covs["first_gap_days"] = df_k5_covs["first_gap_days"].fillna(df_k5_covs["first_gap_days"].median())
df_surv_merged = df_survival.merge(df_k5_covs, on="source_id", how="inner")

# Restrict to non-trivial duration (isolated sources are exactly 0 by construction)
df_cox = df_surv_merged[df_surv_merged["duration_days"] > 0.01].copy()

# First-detection instrument (needed to test/correct for the confound we found)
df_plumes = pd.read_parquet("/kaggle/working/nm_plumes_clean.parquet")
KNOWN_MISMATCH_IDS = ["CH4_1B2_250m_-104.11754_32.05183?status=not_deleted",
                       "CH4_1B2_250m_-104.11947_32.02271?status=not_deleted"]
df_sources_clean = df_sources_full[~df_sources_full["source_name"].isin(KNOWN_MISMATCH_IDS)].copy()
plume_dt_lookup = dict(zip(df_plumes["plume_id"], df_plumes["datetime"]))
plume_instr_lookup = dict(zip(df_plumes["plume_id"], df_plumes["instrument"]))

first_instr = []
for _, row in df_sources_clean.iterrows():
    pids = list(row["plume_ids"]) if isinstance(row["plume_ids"], (list, np.ndarray)) else []
    pid_dates = [(pid, plume_dt_lookup.get(pid)) for pid in pids if pid in plume_dt_lookup]
    if not pid_dates:
        continue
    pid_dates.sort(key=lambda x: x[1])
    first_instr.append({"source_id": row["source_name"], "first_instrument": plume_instr_lookup.get(pid_dates[0][0])})
df_first_instr = pd.DataFrame(first_instr)
df_first_instr["instrument_grouped"] = df_first_instr["first_instrument"].apply(
    lambda x: x if x in ["GAO", "ang", "av3"] else "other_small")

df_cox_instr = df_cox.merge(df_first_instr, on="source_id", how="left")
COX_COVARIATES = ["detect_rate_early", "span_days_early", "first_gap_days"]
instr_dummies = pd.get_dummies(df_cox_instr["instrument_grouped"], prefix="instr", drop_first=True)
model_df = pd.concat([df_cox_instr[["duration_days", "event_observed"] + COX_COVARIATES], instr_dummies], axis=1).dropna()
model_df["instr_av3"] = model_df["instr_av3"].astype(int)  # lifelines needs int, not bool, for strata
REMAINING_COVARIATES = COX_COVARIATES + [c for c in instr_dummies.columns if c != "instr_av3"]

# Final model: instrument as covariate, stratified on instr_av3 (the one PH-violating covariate)
cox_final = CoxPHFitter()
cox_final.fit(model_df[["duration_days", "event_observed"] + REMAINING_COVARIATES + ["instr_av3"]],
              duration_col="duration_days", event_col="event_observed", strata=["instr_av3"])
print("=== Final Cox model (instrument-adjusted, PH-corrected) ===")
print(cox_final.summary[["coef", "exp(coef)", "p"]])
print("\nUnadjusted comparison: detect_rate_early alone gave coef=-0.735, p=0.0017 (confounded; do not report as-is)")

cox_final.check_assumptions(model_df[["duration_days", "event_observed"] + REMAINING_COVARIATES + ["instr_av3"]],
                              p_value_threshold=0.05, show_plots=False)

# Bootstrap CI on concordance index
kf = None
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cph_df = df_cox[["duration_days", "event_observed"] + COX_COVARIATES].copy()
rng = np.random.RandomState(42)
boot_c = []
for _ in range(200):
    idx = rng.choice(len(cph_df), size=len(cph_df), replace=True)
    boot_sample = cph_df.iloc[idx].reset_index(drop=True)
    try:
        m = CoxPHFitter()
        m.fit(boot_sample, duration_col="duration_days", event_col="event_observed")
        pred = m.predict_partial_hazard(boot_sample)
        boot_c.append(concordance_index(boot_sample["duration_days"], -pred, boot_sample["event_observed"]))
    except Exception:
        continue
print(f"\nBootstrap C-index: {np.mean(boot_c):.3f}, 95% CI [{np.percentile(boot_c,2.5):.3f}, {np.percentile(boot_c,97.5):.3f}]")

n_events = model_df["event_observed"].sum()
print(f"\nEvents-per-variable: {n_events} events / 3 covariates = {n_events/3:.1f}")

joblib.dump(cox_final, "/kaggle/working/cox_survival_model.pkl")
model_df.to_parquet("/kaggle/working/cox_model_training_data.parquet")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 9.7 MB/s eta 0:00:00
=== Final Cox model (instrument-adjusted, PH-corrected) ===
                       coef  exp(coef)         p
covariate                                       
detect_rate_early -0.293970   0.745299  0.246804
span_days_early   -0.000136   0.999864  0.232122
first_gap_days     0.000204   1.000204  0.280379
instr_ang         -0.325155   0.722415  0.004572
instr_other_small -0.208592   0.811727  0.647323

Unadjusted comparison: detect_rate_early alone gave coef=-0.735, p=0.0017 (confounded; do not report as-is)
Proportional hazard assumption looks okay.

Bootstrap C-index: 0.564, 95% CI [0.537, 0.593]

Events-per-variable: 445 events / 3 covariates = 148.3


In [8]:
# ============================================================
# PHASE 2 — GHGRP Subpart W pull, entity resolution, MAG taxonomy
# ============================================================
GHGRP_BASE = "https://data.epa.gov/efservice"

url = f"{GHGRP_BASE}/PUB_DIM_FACILITY/STATE/=/NM/JSON"
df_fac = pd.DataFrame(requests.get(url, timeout=60).json())

def reports_subpart_w(subparts_str):
    if pd.isna(subparts_str):
        return False
    return "W" in [s.strip() for s in str(subparts_str).split(",")]

df_w = df_fac[df_fac["reported_subparts"].apply(reports_subpart_w)].copy()
df_w.to_csv("/kaggle/working/nm_ghgrp_subpart_w_facilities.csv", index=False)

all_emissions = []
for fid in df_w["facility_id"].unique():
    resp = requests.get(f"{GHGRP_BASE}/PUB_FACTS_SUBP_GHG_EMISSION/FACILITY_ID/{fid}/JSON", timeout=60)
    if resp.status_code == 200 and resp.text.strip():
        try:
            all_emissions.extend(resp.json())
        except Exception:
            pass
    time.sleep(0.1)
df_emissions = pd.DataFrame(all_emissions)

GWP_CH4 = 25  # pre-2024-amendment GWP; correct for all reporting years present (2011-2023)
df_ch4 = df_emissions[df_emissions["gas_id"] == 2].copy()
df_ch4["ch4_metric_tons"] = df_ch4["co2e_emission"] / GWP_CH4
facility_info_cols = ["facility_id", "facility_name", "latitude", "longitude", "reported_industry_types"]
df_fac_dedup = df_w.sort_values("year").drop_duplicates(subset="facility_id", keep="last")[facility_info_cols]
df_ghgrp_ch4 = df_ch4.merge(df_fac_dedup, on="facility_id", how="left")
df_ghgrp_ch4.to_parquet("/kaggle/working/nm_ghgrp_ch4_clean.parquet")

# Entity resolution: nearest GHGRP facility, haversine distance
from math import radians, sin, cos, sqrt, atan2
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = radians(lat2 - lat1), radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1-a))

df_ghgrp_locs = df_ghgrp_ch4.drop_duplicates(subset="facility_id")[
    ["facility_id", "facility_name", "latitude", "longitude", "reported_industry_types"]].reset_index(drop=True)
from scipy.spatial import cKDTree
tree = cKDTree(df_ghgrp_locs[["latitude", "longitude"]].values)
_, nearest_idx = tree.query(df_sources_full[["latitude", "longitude"]].values, k=1)

df_sources_full["nearest_ghgrp_facility_id"] = df_ghgrp_locs.loc[nearest_idx, "facility_id"].values
df_sources_full["nearest_ghgrp_segment"] = df_ghgrp_locs.loc[nearest_idx, "reported_industry_types"].values
df_sources_full["distance_km"] = [
    haversine_km(r.latitude, r.longitude, df_ghgrp_locs.loc[r.nearest_ghgrp_idx if False else idx, "latitude"],
                 df_ghgrp_locs.loc[idx, "longitude"])
    for idx, r in zip(nearest_idx, df_sources_full.itertuples())
]

def match_confidence(dist_km):
    if dist_km <= 1.0: return "high"
    elif dist_km <= 5.0: return "medium"
    elif dist_km <= 15.0: return "low"
    return "no_match"
df_sources_full["match_confidence"] = df_sources_full["distance_km"].apply(match_confidence)

SITE_RESOLVED_SEGMENTS = ["W-PROC", "W-NGTC", "W-LDC", "W-UNSTG"]
BASIN_AGGREGATED_SEGMENTS = ["W-ONSH", "W-GB"]
def segment_is_site_resolved(segment_str):
    if pd.isna(segment_str): return None
    codes = [s.strip() for s in str(segment_str).split(",")]
    has_site, has_basin = any(c in SITE_RESOLVED_SEGMENTS for c in codes), any(c in BASIN_AGGREGATED_SEGMENTS for c in codes)
    if has_site and not has_basin: return True
    if has_basin and not has_site: return False
    return "mixed"
df_sources_full["nearest_segment_type"] = df_sources_full["nearest_ghgrp_segment"].apply(segment_is_site_resolved)

def assign_mag_class(row):
    if row["match_confidence"] == "no_match": return "no_public_match"
    if row["match_confidence"] == "low": return "ambiguous"
    if row["nearest_segment_type"] is True: return "directly_comparable"
    elif row["nearest_segment_type"] is False: return "aggregation_limited"
    return "ambiguous"
df_sources_full["mag_class"] = df_sources_full.apply(assign_mag_class, axis=1)

print(df_sources_full["mag_class"].value_counts())
df_sources_full.to_parquet("/kaggle/working/nm_sources_final_mag.parquet")

mag_class
ambiguous              820
no_public_match        257
directly_comparable    170
aggregation_limited     45
Name: count, dtype: int64


In [9]:
# ============================================================
# MAG-risk classifier: predict traceability class from early-window features
# ============================================================
df_mag = pd.read_parquet("/kaggle/working/nm_sources_final_mag.parquet")
df_k5 = pd.read_parquet("/kaggle/working/nm_features_k5_v2.parquet")
df_mag_merged = df_k5.merge(
    df_mag[["source_name", "mag_class", "distance_km"]].rename(columns={"source_name": "source_id"}),
    on="source_id", how="inner")

MAG_FEATURE_COLS = ["n_detect_early", "detect_rate_early", "span_days_early", "first_gap_days"]
X_mag = df_mag_merged[MAG_FEATURE_COLS].copy()
X_mag["first_gap_days"] = X_mag["first_gap_days"].fillna(X_mag["first_gap_days"].median())

MAG_CONFIG = FINAL_CONFIG  # same locked hyperparameters as the persistence classifier
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for target_name in ["directly_comparable", "no_public_match"]:
    y_mag = (df_mag_merged["mag_class"] == target_name).astype(int).values
    aurocs, auprcs = [], []
    for tr, te in skf.split(X_mag, y_mag):
        pw = (y_mag[tr] == 0).sum() / max((y_mag[tr] == 1).sum(), 1)
        m = xgb.XGBClassifier(**MAG_CONFIG, scale_pos_weight=pw, tree_method="hist", eval_metric="aucpr", random_state=42)
        m.fit(X_mag.iloc[tr], y_mag[tr])
        proba = m.predict_proba(X_mag.iloc[te])[:, 1]
        aurocs.append(roc_auc_score(y_mag[te], proba)); auprcs.append(average_precision_score(y_mag[te], proba))
    print(f"{target_name}: AUROC={np.mean(aurocs):.3f}±{np.std(aurocs):.3f}, AUPRC={np.mean(auprcs):.3f}±{np.std(auprcs):.3f}")

    # Bootstrap CI
    pw_full = (y_mag == 0).sum() / (y_mag == 1).sum()
    oof = np.zeros(len(y_mag))
    for tr, te in skf.split(X_mag, y_mag):
        pw = (y_mag[tr] == 0).sum() / max((y_mag[tr] == 1).sum(), 1)
        m = xgb.XGBClassifier(**MAG_CONFIG, scale_pos_weight=pw, tree_method="hist", eval_metric="aucpr", random_state=42)
        m.fit(X_mag.iloc[tr], y_mag[tr]); oof[te] = m.predict_proba(X_mag.iloc[te])[:, 1]
    boot_auroc, boot_auprc = [], []
    for _ in range(2000):
        idx = rng.choice(len(y_mag), size=len(y_mag), replace=True)
        if len(set(y_mag[idx])) < 2: continue
        boot_auroc.append(roc_auc_score(y_mag[idx], oof[idx])); boot_auprc.append(average_precision_score(y_mag[idx], oof[idx]))
    print(f"  Bootstrap CI: AUROC [{np.percentile(boot_auroc,2.5):.3f}, {np.percentile(boot_auroc,97.5):.3f}], "
          f"AUPRC [{np.percentile(boot_auprc,2.5):.3f}, {np.percentile(boot_auprc,97.5):.3f}]")

    m_full = xgb.XGBClassifier(**MAG_CONFIG, scale_pos_weight=pw_full, tree_method="hist", eval_metric="aucpr", random_state=42)
    m_full.fit(X_mag, y_mag)
    joblib.dump(m_full, f"/kaggle/working/mag_classifier_{target_name}.pkl")

joblib.dump(MAG_FEATURE_COLS, "/kaggle/working/mag_classifier_feature_order.pkl")

# GHGRP-overlap-period restriction check (only sources first-detected <= 2023 have real reporting-year overlap)
df_plumes = pd.read_parquet("/kaggle/working/nm_plumes_clean.parquet")
plume_dt_lookup = dict(zip(df_plumes["plume_id"], pd.to_datetime(df_plumes["datetime"], utc=True)))
def first_detection_in_overlap(plume_ids):
    if not isinstance(plume_ids, (list, np.ndarray)): return False
    dates = [plume_dt_lookup.get(pid) for pid in plume_ids if pid in plume_dt_lookup]
    dates = [d for d in dates if d is not None]
    return bool(dates) and min(dates).year <= 2023
df_mag["in_overlap"] = df_mag["plume_ids"].apply(first_detection_in_overlap)
print(f"\nSources in GHGRP overlap window: {df_mag['in_overlap'].sum()} / {len(df_mag)}")

directly_comparable: AUROC=0.822±0.039, AUPRC=0.420±0.088
  Bootstrap CI: AUROC [0.782, 0.851], AUPRC [0.306, 0.465]
no_public_match: AUROC=0.798±0.020, AUPRC=0.500±0.043
  Bootstrap CI: AUROC [0.763, 0.821], AUPRC [0.426, 0.556]

Sources in GHGRP overlap window: 801 / 1292


In [10]:
# ============================================================
# Instrument-confound checks + MAG threshold sensitivity
# ============================================================
df_merged_instr = df_k5.merge(df_first_instr[["source_id", "instrument_grouped"]], on="source_id", how="left")
instr_dummies_p = pd.get_dummies(df_merged_instr["instrument_grouped"], prefix="instr", drop_first=True)
X_with_instr = pd.concat([X, instr_dummies_p], axis=1)

def cv_auc_auprc(Xd, yd, config):
    a1, a2 = [], []
    for tr, te in skf.split(Xd, yd):
        pw = (yd[tr] == 0).sum() / max((yd[tr] == 1).sum(), 1)
        m = xgb.XGBClassifier(**config, scale_pos_weight=pw, tree_method="hist", eval_metric="aucpr", random_state=42)
        m.fit(Xd.iloc[tr], yd[tr]); proba = m.predict_proba(Xd.iloc[te])[:, 1]
        a1.append(roc_auc_score(yd[te], proba)); a2.append(average_precision_score(yd[te], proba))
    return np.mean(a1), np.mean(a2)

auroc_no, auprc_no = cv_auc_auprc(X, y, FINAL_CONFIG)
auroc_i, auprc_i = cv_auc_auprc(X_with_instr, y, FINAL_CONFIG)
print(f"Persistence classifier -- without instrument: AUROC={auroc_no:.3f}, AUPRC={auprc_no:.3f}")
print(f"Persistence classifier -- with instrument:    AUROC={auroc_i:.3f}, AUPRC={auprc_i:.3f}")
print("(Modest confounding: core features remain dominant, unlike the survival model's confound)")

# MAG classifier: instrument + overlap-restricted checks
df_mag_instr = df_mag_merged.merge(df_first_instr[["source_id", "instrument_grouped"]], on="source_id", how="left")
mag_instr_dummies = pd.get_dummies(df_mag_instr["instrument_grouped"], prefix="instr", drop_first=True)
X_mag_with_instr = pd.concat([X_mag, mag_instr_dummies], axis=1)
for target_name in ["directly_comparable", "no_public_match"]:
    y_mag = (df_mag_instr["mag_class"] == target_name).astype(int).values
    a1, a2 = cv_auc_auprc(X_mag, y_mag, MAG_CONFIG)
    b1, b2 = cv_auc_auprc(X_mag_with_instr, y_mag, MAG_CONFIG)
    print(f"MAG {target_name} -- without/with instrument: AUROC {a1:.3f}/{b1:.3f}, AUPRC {a2:.3f}/{b2:.3f}")

# MAG distance-threshold sensitivity (5/10/15/20 km)
def assign_mag_class_threshold(row, no_match_km):
    if row["distance_km"] > no_match_km: return "no_public_match"
    if row["match_confidence"] == "low": return "ambiguous"
    if row["nearest_segment_type"] is True: return "directly_comparable"
    elif row["nearest_segment_type"] is False: return "aggregation_limited"
    return "ambiguous"

from scipy.stats import spearmanr
df_ord = df_mag[df_mag["persistence_class"] != "insufficient_evidence"].copy()
persist_order = ["isolated", "intermittent", "persistent", "ultra_persistent"]
df_ord["persistence_rank"] = df_ord["persistence_class"].map({c: i for i, c in enumerate(persist_order)})
for threshold_km in [5, 10, 15, 20]:
    classes = df_ord.apply(lambda r: assign_mag_class_threshold(r, threshold_km), axis=1)
    has_match = (classes != "no_public_match").astype(int)
    rho, p = spearmanr(df_ord["persistence_rank"], has_match)
    print(f"Threshold {threshold_km}km: rho={rho:.3f}, p={p:.4f}, no_public_match n={(classes=='no_public_match').sum()}")
print("NOTE: this association is not robust across thresholds -- report as inconclusive, not a headline finding.")

Persistence classifier -- without instrument: AUROC=0.763, AUPRC=0.575
Persistence classifier -- with instrument:    AUROC=0.790, AUPRC=0.588
(Modest confounding: core features remain dominant, unlike the survival model's confound)
MAG directly_comparable -- without/with instrument: AUROC 0.822/0.826, AUPRC 0.420/0.425
MAG no_public_match -- without/with instrument: AUROC 0.798/0.798, AUPRC 0.500/0.501
Threshold 5km: rho=0.061, p=0.0413, no_public_match n=940
Threshold 10km: rho=0.030, p=0.3216, no_public_match n=580
Threshold 15km: rho=0.073, p=0.0149, no_public_match n=224
Threshold 20km: rho=-0.012, p=0.6840, no_public_match n=40
NOTE: this association is not robust across thresholds -- report as inconclusive, not a headline finding.


In [11]:
# ============================================================
# ENVIRONMENTAL JUSTICE / STEP 1 — Census demographics + block-group polygons
# ============================================================
CENSUS_API_KEY = UserSecretsClient().get_secret("CENSUS_API_KEY")  # add this secret first

acs_url = "https://api.census.gov/data/2023/acs/acs5"
acs_vars = "NAME,B01003_001E,C17002_002E,C17002_003E,B03002_003E,B03002_012E,B25044_003E,B25044_010E"
params = {"get": acs_vars, "for": "block group:*", "in": "state:35 county:* tract:*", "key": CENSUS_API_KEY}
resp = requests.get(acs_url, params=params, timeout=60)
acs_data = resp.json()
df_acs_bg = pd.DataFrame(acs_data[1:], columns=acs_data[0])

numeric_cols = ["B01003_001E", "C17002_002E", "C17002_003E", "B03002_003E", "B03002_012E", "B25044_003E", "B25044_010E"]
for col in numeric_cols:
    df_acs_bg[col] = pd.to_numeric(df_acs_bg[col], errors="coerce")

df_acs_bg["total_pop"] = df_acs_bg["B01003_001E"]
df_acs_bg["pct_poverty_proxy"] = (df_acs_bg["C17002_002E"] + df_acs_bg["C17002_003E"]) / df_acs_bg["total_pop"] * 100
df_acs_bg["pct_minority"] = (1 - df_acs_bg["B03002_003E"] / df_acs_bg["total_pop"]) * 100
df_acs_bg["pct_hispanic"] = df_acs_bg["B03002_012E"] / df_acs_bg["total_pop"] * 100
df_acs_bg["pct_no_vehicle"] = (df_acs_bg["B25044_003E"] + df_acs_bg["B25044_010E"]) / df_acs_bg["total_pop"] * 100
df_acs_bg["GEOID"] = df_acs_bg["state"] + df_acs_bg["county"] + df_acs_bg["tract"] + df_acs_bg["block group"]
df_acs_bg.to_csv("/kaggle/working/nm_acs_blockgroups_final.csv", index=False)

# TIGERweb block-group polygons (real geometry, needed for area-weighted overlap)
!pip install geopandas --quiet
import geopandas as gpd

tigerweb_url = "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/Tracts_Blocks/MapServer/1/query"
all_features = []
offset = 0
while True:
    params = {"where": "STATE='35'", "outFields": "GEOID", "f": "geojson",
              "resultOffset": offset, "resultRecordCount": 500}
    resp = requests.get(tigerweb_url, params=params, timeout=60)
    feats = resp.json().get("features", [])
    if not feats:
        break
    all_features.extend(feats)
    offset += 500
    if len(feats) < 500:
        break

gdf_bg = gpd.GeoDataFrame.from_features(all_features, crs="EPSG:4326")
gdf_bg["GEOID"] = gdf_bg["GEOID"].astype(str).str.zfill(12)
df_acs_bg["GEOID"] = df_acs_bg["GEOID"].str.zfill(12)
gdf_bg = gdf_bg.merge(df_acs_bg[["GEOID", "total_pop", "pct_minority", "pct_poverty_proxy",
                                    "pct_hispanic", "pct_no_vehicle"]], on="GEOID", how="inner")

gdf_bg_proj = gdf_bg.to_crs("EPSG:32613")  # UTM 13N, for accurate km-based buffering
gdf_bg_proj["bg_area_m2"] = gdf_bg_proj.geometry.area
gdf_bg_proj.to_parquet("/kaggle/working/nm_blockgroups_polygons.parquet")
print(f"Block groups with geometry + demographics: {len(gdf_bg_proj)}")

Block groups with geometry + demographics: 1614


In [12]:
# ============================================================
# ENVIRONMENTAL JUSTICE / STEP 2 — Area-weighted buffer exposure
# (validated: this method replaced two failed alternatives -- tract nearest-join
#  and block-group centroid-in-radius -- both of which broke down in this rural geography)
# ============================================================
from shapely.geometry import Point

gdf_bg_proj = gpd.read_parquet("/kaggle/working/nm_blockgroups_polygons.parquet")
df_sources_full = pd.read_parquet("/kaggle/working/nm_sources_final_mag.parquet")
df_ordered = df_sources_full[df_sources_full["persistence_class"] != "insufficient_evidence"].copy()
persist_order = ["isolated", "intermittent", "persistent", "ultra_persistent"]
df_ordered["persistence_rank"] = df_ordered["persistence_class"].map({c: i for i, c in enumerate(persist_order)})

gdf_ordered = gpd.GeoDataFrame(
    df_ordered, geometry=[Point(lon, lat) for lon, lat in zip(df_ordered["longitude"], df_ordered["latitude"])],
    crs="EPSG:4326").to_crs("EPSG:32613")

DEMO_COLS = ["pct_minority", "pct_poverty_proxy", "pct_hispanic", "pct_no_vehicle"]

def compute_buffer_exposure(gdf_src, gdf_bg, radius_km):
    radius_m = radius_km * 1000
    sindex = gdf_bg.sindex
    results = []
    for geom in gdf_src.geometry:
        buf = geom.buffer(radius_m)
        possible_idx = list(sindex.intersection(buf.bounds))
        if not possible_idx:
            results.append({col: np.nan for col in DEMO_COLS} | {"total_pop_exposed": 0}); continue
        candidates = gdf_bg.iloc[possible_idx]
        overlap_frac = (candidates.geometry.intersection(buf).area / candidates["bg_area_m2"]).clip(0, 1)
        touched = overlap_frac > 0
        if touched.sum() == 0:
            results.append({col: np.nan for col in DEMO_COLS} | {"total_pop_exposed": 0}); continue
        weights = candidates.loc[touched, "total_pop"] * overlap_frac[touched]
        row = {col: np.average(candidates.loc[touched, col], weights=weights) for col in DEMO_COLS}
        row["total_pop_exposed"] = weights.sum()
        results.append(row)
    return pd.DataFrame(results)

for radius_km in [1, 2, 5, 10, 15]:
    exposure = compute_buffer_exposure(gdf_ordered, gdf_bg_proj, radius_km)
    combined = pd.concat([df_ordered.reset_index(drop=True), exposure], axis=1)
    n_zero = (combined["total_pop_exposed"] == 0).sum()
    print(f"{radius_km}km: n={len(combined)}, zero-coverage={n_zero}")
    combined.to_parquet(f"/kaggle/working/nm_buffer_exposure_{radius_km}km.parquet")

1km: n=1120, zero-coverage=2
2km: n=1120, zero-coverage=1
5km: n=1120, zero-coverage=0
10km: n=1120, zero-coverage=0
15km: n=1120, zero-coverage=0


In [13]:
# ============================================================
# ENVIRONMENTAL JUSTICE / STEP 3 — Robustness: FDR, clustering, spatial autocorrelation
# ============================================================
!pip install libpysal esda statsmodels --quiet
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
from libpysal.weights import KNN
from esda.moran import Moran

test_rows = []
for radius_km in [1, 2, 5, 10, 15]:
    dfb = pd.read_parquet(f"/kaggle/working/nm_buffer_exposure_{radius_km}km.parquet")
    for col, label in zip(DEMO_COLS, ["% minority", "% below poverty", "% Hispanic", "% no vehicle"]):
        rho, p = spearmanr(dfb["persistence_rank"], dfb[col], nan_policy="omit")
        test_rows.append({"radius_km": radius_km, "variable": label, "rho": rho, "p_raw": p})
df_tests = pd.DataFrame(test_rows)
_, p_fdr, _, _ = multipletests(df_tests["p_raw"], alpha=0.05, method="fdr_bh")
df_tests["p_fdr"] = p_fdr
df_tests["significant_after_fdr"] = df_tests["p_fdr"] < 0.05
print(df_tests.to_string(index=False))
print(f"\nSignificant after FDR: {df_tests['significant_after_fdr'].sum()} / {len(df_tests)}")
df_tests.to_csv("/kaggle/working/supplementary_EJ_tests_FDR.csv", index=False)

# Cluster-robust regression (5km, clustered by nearest block group)
tree_bg = cKDTree(gdf_bg_proj.geometry.centroid.get_coordinates().values)
_, nearest_bg_idx = tree_bg.query(gdf_ordered.geometry.get_coordinates().values, k=1)
df_ordered["nearest_bg_geoid"] = gdf_bg_proj.iloc[nearest_bg_idx]["GEOID"].values

df_5km = pd.read_parquet("/kaggle/working/nm_buffer_exposure_5km.parquet")
df_5km = df_5km.merge(df_ordered[["source_name", "nearest_bg_geoid"]], on="source_name", how="left")
df_5km_clean = df_5km.dropna(subset=["pct_poverty_proxy", "nearest_bg_geoid"]).copy()

model_naive = smf.ols("persistence_rank ~ pct_poverty_proxy", data=df_5km_clean).fit()
model_cluster = smf.ols("persistence_rank ~ pct_poverty_proxy", data=df_5km_clean).fit(
    cov_type="cluster", cov_kwds={"groups": df_5km_clean["nearest_bg_geoid"]})
print(f"\nNaive SE: coef={model_naive.params['pct_poverty_proxy']:.5f}, p={model_naive.pvalues['pct_poverty_proxy']:.4f}")
print(f"Cluster-robust SE: coef={model_cluster.params['pct_poverty_proxy']:.5f}, p={model_cluster.pvalues['pct_poverty_proxy']:.4f}")

# Moran's I on regression residuals
df_5km_clean["residual"] = model_naive.resid
coords = df_5km_clean[["longitude", "latitude"]].values
w = KNN.from_array(coords, k=8); w.transform = "r"
moran = Moran(df_5km_clean["residual"].values, w)
print(f"\nMoran's I: {moran.I:.4f}, p={moran.p_sim:.4f} (not significant = no spatial confounding)")

 radius_km        variable      rho    p_raw    p_fdr  significant_after_fdr
         1      % minority 0.083179 0.005387 0.011971                   True
         1 % below poverty 0.104639 0.000458 0.002354                   True
         1      % Hispanic 0.080093 0.007376 0.014753                   True
         1    % no vehicle 0.104417 0.000471 0.002354                   True
         2      % minority 0.063878 0.032630 0.038388                   True
         2 % below poverty 0.105955 0.000385 0.002354                   True
         2      % Hispanic 0.062157 0.037623 0.041803                   True
         2    % no vehicle 0.091478 0.002191 0.007302                   True
         5      % minority 0.084525 0.004645 0.011613                   True
         5 % below poverty 0.104319 0.000471 0.002354                   True
         5      % Hispanic 0.074643 0.012464 0.019175                   True
         5    % no vehicle 0.095580 0.001362 0.005450                   True

In [14]:
# ============================================================
# ENVIRONMENTAL JUSTICE / STEP 4 — WorldPop validation of the uniform-density assumption
# ============================================================
!pip install rasterio rasterstats --quiet
import rasterio
from rasterio.windows import from_bounds
from rasterstats import zonal_stats
from shapely.geometry import box

NM_BBOX = (-104.6, 32.0, -103.0, 32.8)
WORLDPOP_URL = ("https://worldpop-public-data.soton.ac.uk/GIS/Population/"
                "Global_2000_2020_Constrained/2020/BSGM/USA/usa_ppp_2020_UNadj_constrained.tif")
LOCAL_PATH = "/kaggle/working/usa_ppp_2020_UNadj_constrained.tif"

# Full download (this server does not support HTTP range requests, so windowed reads fail)
with requests.get(WORLDPOP_URL, stream=True, timeout=600) as r:
    r.raise_for_status()
    with open(LOCAL_PATH, "wb") as f:
        for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
            f.write(chunk)

with rasterio.open(LOCAL_PATH) as src:
    window = from_bounds(*NM_BBOX, transform=src.transform)
    pop_array = src.read(1, window=window)
    window_transform = src.window_transform(window)
    nodata = src.nodata
pop_array_clean = np.where(pop_array == nodata, 0, pop_array)
import os
os.remove(LOCAL_PATH)  # free disk space once the small window is extracted

gdf_bg_4326 = gdf_bg_proj.to_crs("EPSG:4326")
NM_BBOX_GEOM = box(*NM_BBOX)
gdf_bg_in_window = gdf_bg_4326[gdf_bg_4326.geometry.intersects(NM_BBOX_GEOM)].copy()  # restrict scope correctly

stats = zonal_stats(gdf_bg_in_window.geometry, pop_array_clean, affine=window_transform, stats=["sum"], nodata=0)
gdf_bg_in_window["raster_total_pop"] = [s["sum"] if s["sum"] is not None else 0 for s in stats]
rho_val, p_val = spearmanr(gdf_bg_in_window["total_pop"], gdf_bg_in_window["raster_total_pop"])
print(f"ACS vs WorldPop validation (n={len(gdf_bg_in_window)} in-window block groups): rho={rho_val:.3f}, p={p_val:.6f}")

# Recompute 5km exposure with raster weighting instead of uniform density, compare
sindex_bg = gdf_bg_4326.sindex
buffers_proj = gdf_ordered.geometry.buffer(5000)
buffers_4326 = gpd.GeoSeries(buffers_proj, crs="EPSG:32613").to_crs("EPSG:4326")
raster_results = []
for buf in buffers_4326:
    possible_idx = list(sindex_bg.intersection(buf.bounds))
    if not possible_idx:
        raster_results.append({c: np.nan for c in DEMO_COLS}); continue
    candidates = gdf_bg_4326.iloc[possible_idx]
    intersections = candidates.geometry.intersection(buf)
    touched = ~intersections.is_empty
    if touched.sum() == 0:
        raster_results.append({c: np.nan for c in DEMO_COLS}); continue
    intersect_stats = zonal_stats(list(intersections[touched]), pop_array_clean, affine=window_transform,
                                    stats=["sum"], nodata=0)
    weights = np.array([s["sum"] if s["sum"] is not None else 0 for s in intersect_stats])
    if weights.sum() == 0:
        raster_results.append({c: np.nan for c in DEMO_COLS}); continue
    touched_c = candidates[touched]
    raster_results.append({c: np.average(touched_c[c], weights=weights) for c in DEMO_COLS})

df_raster = pd.concat([df_ordered.reset_index(drop=True), pd.DataFrame(raster_results)], axis=1)
for col, label in zip(DEMO_COLS, ["% minority", "% below poverty", "% Hispanic", "% no vehicle"]):
    rho_r, p_r = spearmanr(df_raster["persistence_rank"], df_raster[col], nan_policy="omit")
    print(f"{label}: raster-weighted rho={rho_r:.3f}, p={p_r:.4f}")
print("(Compare to uniform-density results in nm_buffer_exposure_5km.parquet -- validated: materially unchanged)")

ACS vs WorldPop validation (n=79 in-window block groups): rho=0.451, p=0.000030
% minority: raster-weighted rho=0.136, p=0.0047
% below poverty: raster-weighted rho=0.106, p=0.0274
% Hispanic: raster-weighted rho=0.135, p=0.0048
% no vehicle: raster-weighted rho=0.095, p=0.0490
(Compare to uniform-density results in nm_buffer_exposure_5km.parquet -- validated: materially unchanged)


In [15]:
# ============================================================
# MITIGATION SIMULATION — rank-based scoring (final, corrected version;
# an earlier multiplicative scoring attempt let unbounded emission magnitude
# dominate all other weights -- fixed by converting everything to percentile ranks first)
# ============================================================
df_master = df_sources_full.copy()
df_plumes = pd.read_parquet("/kaggle/working/nm_plumes_clean.parquet")
plume_emission_lookup = df_plumes.set_index("plume_id")["emission_auto"].to_dict()

def mean_source_emission(plume_ids):
    if not isinstance(plume_ids, (list, np.ndarray)): return np.nan
    vals = [plume_emission_lookup.get(pid) for pid in plume_ids]
    vals = [v for v in vals if v is not None and not pd.isna(v)]
    return np.mean(vals) if vals else np.nan

df_master["emission_rate"] = df_master["plume_ids"].apply(mean_source_emission)
df_master["emission_rate"] = df_master.groupby("persistence_class")["emission_rate"].transform(lambda x: x.fillna(x.median()))

# Merge 5km exposure (recompute for ALL 1292 sources, including insufficient_evidence ones --
# mitigation prioritization needs every real source, not just those with a reliable persistence label)
gdf_master_pts = gpd.GeoDataFrame(
    df_master, geometry=[Point(lon, lat) for lon, lat in zip(df_master["longitude"], df_master["latitude"])],
    crs="EPSG:4326").to_crs("EPSG:32613")
exposure_all = compute_buffer_exposure(gdf_master_pts, gdf_bg_proj, 5)
df_master = pd.concat([df_master.reset_index(drop=True), exposure_all], axis=1)

vuln_cols = ["pct_minority", "pct_poverty_proxy", "pct_hispanic", "pct_no_vehicle"]
for col in vuln_cols:
    df_master[f"{col}_z"] = (df_master[col] - df_master[col].mean()) / df_master[col].std()
df_master["vulnerability_index"] = df_master[[f"{c}_z" for c in vuln_cols]].mean(axis=1)
df_master["vulnerability_index"] = (df_master["vulnerability_index"] - df_master["vulnerability_index"].min()) / \
    (df_master["vulnerability_index"].max() - df_master["vulnerability_index"].min())

MAG_WEIGHTS = {"directly_comparable": 1.0, "aggregation_limited": 1.15, "ambiguous": 1.25, "no_public_match": 1.5}
df_master["mag_weight"] = df_master["mag_class"].map(MAG_WEIGHTS)
df_master["persistence_weight"] = df_master["persistence"].fillna(0)

df_master["log_emission"] = np.log1p(df_master["emission_rate"])
df_master["emission_rank"] = df_master["log_emission"].rank(pct=True)
df_master["persistence_rank_pct"] = df_master["persistence_weight"].rank(pct=True)
df_master["mag_rank_pct"] = df_master["mag_weight"].rank(pct=True)
df_master["vulnerability_rank_pct"] = df_master["vulnerability_index"].rank(pct=True)

WEIGHT_SCHEMES = {
    "S1_emissions_only":         {"emission": 1.00, "persistence": 0.00, "mag": 0.00, "vulnerability": 0.00},
    "S2_persistence_aware":      {"emission": 0.60, "persistence": 0.40, "mag": 0.00, "vulnerability": 0.00},
    "S3_accountability_aware":   {"emission": 0.50, "persistence": 0.30, "mag": 0.20, "vulnerability": 0.00},
    "S4_justice_aware":          {"emission": 0.40, "persistence": 0.25, "mag": 0.15, "vulnerability": 0.20},
}
def compute_score(df, w):
    return (w["emission"] * df["emission_rank"] + w["persistence"] * df["persistence_rank_pct"] +
            w["mag"] * df["mag_rank_pct"] + w["vulnerability"] * df["vulnerability_rank_pct"])

BUDGET_LEVELS = [0.05, 0.10, 0.20, 0.30]
results = []
for name, w in WEIGHT_SCHEMES.items():
    df_master[f"score_{name}"] = compute_score(df_master, w)
    ranked = df_master.sort_values(f"score_{name}", ascending=False).reset_index(drop=True)
    for budget in BUDGET_LEVELS:
        n_sel = int(len(ranked) * budget)
        sel = ranked.iloc[:n_sel]
        pct_em = 100 * sel["emission_rate"].sum() / df_master["emission_rate"].sum()
        pct_persist = 100 * sel["persistence_class"].isin(["persistent", "ultra_persistent"]).sum() / \
            df_master["persistence_class"].isin(["persistent", "ultra_persistent"]).sum()
        pct_nomatch = 100 * (sel["mag_class"] == "no_public_match").sum() / (df_master["mag_class"] == "no_public_match").sum()
        vuln_b = (sel["total_pop_exposed"] * sel["vulnerability_index"]).sum()
        total_vuln = (df_master["total_pop_exposed"] * df_master["vulnerability_index"]).sum()
        pct_vuln = 100 * vuln_b / total_vuln
        results.append({"strategy": name, "budget_pct": int(budget*100), "pct_emissions_captured": round(pct_em,1),
                         "pct_persistent_captured": round(pct_persist,1), "pct_high_mag_captured": round(pct_nomatch,1),
                         "pct_vulnerability_benefit": round(pct_vuln,1)})
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
df_results.to_csv("/kaggle/working/mitigation_simulation_results_v2.csv", index=False)
df_master.to_parquet("/kaggle/working/nm_mitigation_master_complete.parquet")

# Weight-sensitivity check: 5 alternative schemes, does S4-like justice-weighting beat S1 everywhere?
ALT_SCHEMES = {**WEIGHT_SCHEMES,
    "equal_weights":   {"emission": 0.25, "persistence": 0.25, "mag": 0.25, "vulnerability": 0.25},
    "emission_heavy":  {"emission": 0.55, "persistence": 0.15, "mag": 0.10, "vulnerability": 0.20},
    "justice_heavy":   {"emission": 0.25, "persistence": 0.15, "mag": 0.10, "vulnerability": 0.50},
    "no_mag":          {"emission": 0.45, "persistence": 0.30, "mag": 0.00, "vulnerability": 0.25},
}
s1_ref = {b: [r for r in results if r["strategy"]=="S1_emissions_only" and r["budget_pct"]==int(b*100)][0] for b in BUDGET_LEVELS}
print("\nWeight-sensitivity: does each scheme beat S1 on vulnerability capture at every budget?")
for name, w in ALT_SCHEMES.items():
    if name in WEIGHT_SCHEMES: continue
    df_master[f"score_{name}"] = compute_score(df_master, w)
    beats_all = True
    for budget in BUDGET_LEVELS:
        ranked = df_master.sort_values(f"score_{name}", ascending=False).reset_index(drop=True)
        n_sel = int(len(ranked) * budget)
        sel = ranked.iloc[:n_sel]
        vuln_b = (sel["total_pop_exposed"] * sel["vulnerability_index"]).sum()
        total_vuln = (df_master["total_pop_exposed"] * df_master["vulnerability_index"]).sum()
        pct_vuln = 100 * vuln_b / total_vuln
        if pct_vuln <= s1_ref[budget]["pct_vulnerability_benefit"]:
            beats_all = False
    print(f"  {name}: beats S1 at every budget = {beats_all}")

               strategy  budget_pct  pct_emissions_captured  pct_persistent_captured  pct_high_mag_captured  pct_vulnerability_benefit
      S1_emissions_only           5                    24.4                      1.3                    5.4                        1.3
      S1_emissions_only          10                    35.9                      6.7                    7.8                       14.0
      S1_emissions_only          20                    52.4                     20.0                   20.6                       25.6
      S1_emissions_only          30                    64.1                     32.0                   29.6                       32.6
   S2_persistence_aware           5                    11.7                     20.7                    5.1                       13.0
   S2_persistence_aware          10                    20.2                     36.7                    8.9                       30.8
   S2_persistence_aware          20                    

In [16]:
# ============================================================
# ROBUSTNESS — class stability (kappa), complete-case ablation, calibration plot
# ============================================================
def cohens_kappa(a, b):
    labels = sorted(set(a) | set(b))
    po = (a.values == b.values).mean()
    pe = sum(((a == l).mean()) * ((b == l).mean()) for l in labels)
    return (po - pe) / (1 - pe)

for variant, label in [("persistence_class_sens_low", "min_reliable=2"),
                        ("persistence_class_sens_high", "min_reliable=10"),
                        ("persistence_class_sens_3", "min_reliable=3"),
                        ("persistence_class_sens_15", "min_reliable=15")]:
    k = cohens_kappa(df_sources_full["persistence_class"], df_sources_full[variant])
    print(f"Kappa (primary vs {label}): {k:.3f}")

# Complete-case (non-imputed) emission ablation
EMISSION_COLS = ["first_emission_early", "max_emission_early", "mean_emission_early"]
complete_mask = df_k5[EMISSION_COLS].notna().all(axis=1)
df_cc = df_k5[complete_mask].copy()
X_cc = df_cc[["n_obs_early","n_detect_early","detect_rate_early","span_days_early","first_gap_days"] + EMISSION_COLS + ["n_emission_readings_early"]]
y_cc = df_cc["persistence_class"].isin(["persistent","ultra_persistent"]).astype(int).values
print(f"\nComplete-case: n={len(df_cc)} ({100*len(df_cc)/len(df_k5):.1f}%), positive rate={y_cc.mean():.3f} "
      f"(vs {y.mean():.3f} full sample -- missingness is NOT random, report as limitation)")

# Calibration curve plot
from sklearn.calibration import calibration_curve
oof_proba = np.zeros(len(y))
for tr, te in skf.split(X, y):
    pw = (y[tr]==0).sum()/max((y[tr]==1).sum(),1)
    cal = CalibratedClassifierCV(xgb.XGBClassifier(**FINAL_CONFIG, scale_pos_weight=pw, tree_method="hist",
                                                     eval_metric="aucpr", random_state=42), method="isotonic", cv=3)
    cal.fit(X.iloc[tr], y[tr]); oof_proba[te] = cal.predict_proba(X.iloc[te])[:, 1]
frac_pos, mean_pred = calibration_curve(y, oof_proba, n_bins=10, strategy="quantile")
print("\nCalibration curve (predicted, observed):")
for mp, fp in zip(mean_pred, frac_pos):
    print(f"  {mp:.3f}, {fp:.3f}")

Kappa (primary vs min_reliable=2): 0.776
Kappa (primary vs min_reliable=10): 0.827
Kappa (primary vs min_reliable=3): 0.822
Kappa (primary vs min_reliable=15): 0.714

Complete-case: n=366 (32.7%), positive rate=0.273 (vs 0.134 full sample -- missingness is NOT random, report as limitation)

Calibration curve (predicted, observed):
  0.035, 0.062
  0.060, 0.056
  0.068, 0.079
  0.072, 0.034
  0.074, 0.030
  0.080, 0.081
  0.084, 0.075
  0.089, 0.110
  0.168, 0.144
  0.676, 0.670


In [17]:
# ============================================================
# FIGURES — all 7, matching the manuscript, 600 DPI
# ============================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from lifelines import KaplanMeierFitter

FIGDIR = "/kaggle/working/figures"
import os; os.makedirs(FIGDIR, exist_ok=True)
DPI = 600
plt.rcParams.update({"font.size": 11, "figure.dpi": DPI, "savefig.dpi": DPI,
                      "axes.spines.top": False, "axes.spines.right": False})
PALETTE = {"isolated":"#B0B0B0","intermittent":"#5DA5DA","persistent":"#FAA43A",
           "ultra_persistent":"#D62728","insufficient_evidence":"#E8E8E8"}
CLASS_ORDER = ["insufficient_evidence","isolated","intermittent","persistent","ultra_persistent"]
mag_order = ["directly_comparable","aggregation_limited","ambiguous","no_public_match"]
mag_colors = ["#2CA02C","#8C6BB1","#BDBDBD","#D62728"]

# Fig 1: study region
fig, ax = plt.subplots(figsize=(6.5,6))
for cls in CLASS_ORDER:
    sub = df_sources_full[df_sources_full["persistence_class"]==cls]
    ax.scatter(sub["longitude"], sub["latitude"], s=14, alpha=0.75, c=PALETTE[cls],
               label=f"{cls.replace('_',' ')} (n={len(sub)})", edgecolors="none")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.legend(fontsize=8.5); ax.set_aspect("equal")
plt.tight_layout(); plt.savefig(f"{FIGDIR}/fig1_study_region.png", dpi=DPI, bbox_inches="tight"); plt.close()

# Fig 2: class distributions
fig, axes = plt.subplots(1,2, figsize=(11,4.5))
counts = df_sources_full["persistence_class"].value_counts().reindex(CLASS_ORDER)
axes[0].bar(range(len(counts)), counts.values, color=[PALETTE[c] for c in counts.index], edgecolor="black")
axes[0].set_xticks(range(len(counts))); axes[0].set_xticklabels([c.replace("_","\n") for c in counts.index], fontsize=8.5)
mag_counts = df_sources_full["mag_class"].value_counts().reindex(mag_order)
axes[1].bar(range(len(mag_counts)), mag_counts.values, color=mag_colors, edgecolor="black")
axes[1].set_xticks(range(len(mag_counts))); axes[1].set_xticklabels([c.replace("_","\n") for c in mag_counts.index], fontsize=8.5)
plt.tight_layout(); plt.savefig(f"{FIGDIR}/fig2_class_distributions.png", dpi=DPI, bbox_inches="tight"); plt.close()

# Fig 3: Sankey (sigmoid-ribbon method -- validated fix for the node-width units bug)
df_ord2 = df_sources_full[df_sources_full["persistence_class"]!="insufficient_evidence"].copy()
persist_order_s = ["ultra_persistent","persistent","intermittent","isolated"]
flow = df_ord2.groupby(["persistence_class","mag_class"]).size().reset_index(name="count")
left_colors = {"isolated":"#B0B0B0","intermittent":"#5DA5DA","persistent":"#FAA43A","ultra_persistent":"#D62728"}
right_colors_map = dict(zip(mag_order, mag_colors))
def node_heights(order, col): return {c: flow[flow[col]==c]["count"].sum() for c in order}
left_h, right_h = node_heights(persist_order_s,"persistence_class"), node_heights(mag_order,"mag_class")
GAP = df_ord2.shape[0]*0.03
def stack_positions(order, heights):
    pos, yv = {}, 0
    for c in order: pos[c]=(yv, yv+heights[c]); yv += heights[c]+GAP
    return pos, yv-GAP
left_pos, left_total = stack_positions(persist_order_s, left_h)
right_pos, right_total = stack_positions(mag_order, right_h)
max_total = max(left_total, right_total)
left_offset, right_offset = (max_total-left_total)/2, (max_total-right_total)/2
fig, ax = plt.subplots(figsize=(9.5,6.5))
X_LEFT, X_RIGHT = 0, 10
NODE_W = (X_RIGHT-X_LEFT)*0.025  # width in X-axis units -- NOT count units (this was the earlier bug)
N_PTS = 60
left_cursor = {c: left_pos[c][0]+left_offset for c in persist_order_s}
right_cursor = {c: right_pos[c][0]+right_offset for c in mag_order}
for _, row in flow.sort_values("count", ascending=False).iterrows():
    lc, rc, val = row["persistence_class"], row["mag_class"], row["count"]
    y0_l = left_cursor[lc]; y1_l = y0_l+val; left_cursor[lc]=y1_l
    y0_r = right_cursor[rc]; y1_r = y0_r+val; right_cursor[rc]=y1_r
    xs = np.linspace(X_LEFT+NODE_W, X_RIGHT-NODE_W, N_PTS)
    t = np.linspace(0,1,N_PTS); s = 1/(1+np.exp(-12*(t-0.5)))
    top = y0_l+(y0_r-y0_l)*s; bot = y1_l+(y1_r-y1_l)*s
    ax.fill(np.concatenate([xs,xs[::-1]]), np.concatenate([top,bot[::-1]]), color=left_colors[lc], alpha=0.42, linewidth=0)
for c in persist_order_s:
    y0,y1 = left_pos[c][0]+left_offset, left_pos[c][1]+left_offset
    ax.add_patch(mpatches.Rectangle((X_LEFT,y0), NODE_W, y1-y0, facecolor=left_colors[c], edgecolor="black"))
    ax.text(X_LEFT-0.25,(y0+y1)/2, f"{c.replace('_',' ').title()}\n(n={left_h[c]})", ha="right", va="center", fontsize=9.5)
for c in mag_order:
    y0,y1 = right_pos[c][0]+right_offset, right_pos[c][1]+right_offset
    ax.add_patch(mpatches.Rectangle((X_RIGHT-NODE_W,y0), NODE_W, y1-y0, facecolor=right_colors_map[c], edgecolor="black"))
    ax.text(X_RIGHT+0.25,(y0+y1)/2, f"{c.replace('_',' ').title()}\n(n={right_h[c]})", ha="left", va="center", fontsize=9.5)
ax.set_xlim(-2.8,12.8); ax.set_ylim(-max_total*0.03, max_total*1.03); ax.axis("off")
plt.tight_layout(); plt.savefig(f"{FIGDIR}/fig3_sankey.png", dpi=DPI, bbox_inches="tight"); plt.close()

# Fig 4: ROC/PR
from sklearn.metrics import roc_curve, precision_recall_curve, auc
fpr, tpr, _ = roc_curve(y, oof_proba); roc_auc = auc(fpr, tpr)
baseline_proba = X["detect_rate_early"].values
fpr_b, tpr_b, _ = roc_curve(y, baseline_proba); roc_auc_b = auc(fpr_b, tpr_b)
prec, rec, _ = precision_recall_curve(y, oof_proba); pr_auc = average_precision_score(y, oof_proba)
prec_b, rec_b, _ = precision_recall_curve(y, baseline_proba); pr_auc_b = average_precision_score(y, baseline_proba)
fig, axes = plt.subplots(1,2, figsize=(11,4.6))
axes[0].plot(fpr,tpr,color="#D62728",lw=2.2,label=f"XGBoost (AUROC={roc_auc:.3f})")
axes[0].plot(fpr_b,tpr_b,color="#5DA5DA",lw=1.6,ls="--",label=f"Baseline (AUROC={roc_auc_b:.3f})")
axes[0].plot([0,1],[0,1],color="gray",lw=1,ls=":"); axes[0].legend(fontsize=9)
axes[1].plot(rec,prec,color="#D62728",lw=2.2,label=f"XGBoost (AUPRC={pr_auc:.3f})")
axes[1].plot(rec_b,prec_b,color="#5DA5DA",lw=1.6,ls="--",label=f"Baseline (AUPRC={pr_auc_b:.3f})")
axes[1].axhline(y.mean(),color="gray",lw=1,ls=":"); axes[1].legend(fontsize=9)
plt.tight_layout(); plt.savefig(f"{FIGDIR}/fig4_roc_pr.png", dpi=DPI, bbox_inches="tight"); plt.close()

# Fig 5: Kaplan-Meier
df_surv_lbl = df_survival.merge(df_sources_full[["source_name","persistence_class"]].rename(columns={"source_name":"source_id"}), on="source_id", how="left")
df_nt = df_surv_lbl[df_surv_lbl["duration_days"]>0.01].copy()
fig, ax = plt.subplots(figsize=(7.5,5.5))
km_colors = {"intermittent":"#5DA5DA","persistent":"#FAA43A","ultra_persistent":"#D62728"}
for cls in ["intermittent","persistent","ultra_persistent"]:
    sub = df_nt[df_nt["persistence_class"]==cls]
    if len(sub)<3: continue
    kmf = KaplanMeierFitter(); kmf.fit(sub["duration_days"], event_observed=sub["event_observed"], label=f"{cls} (n={len(sub)})")
    kmf.plot_survival_function(ax=ax, color=km_colors[cls], ci_show=True, linewidth=2)
plt.tight_layout(); plt.savefig(f"{FIGDIR}/fig5_kaplan_meier.png", dpi=DPI, bbox_inches="tight"); plt.close()

# Fig 6: EJ correlations across radii
rows=[]
for km in [2,5,10]:
    dfb = pd.read_parquet(f"/kaggle/working/nm_buffer_exposure_{km}km.parquet")
    for col,label in zip(DEMO_COLS, ["% minority","% below poverty","% Hispanic","% no vehicle"]):
        rho,_ = spearmanr(dfb["persistence_rank"], dfb[col], nan_policy="omit")
        rows.append({"radius_km":km,"variable":label,"rho":rho})
df_ej = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(7.5,5))
for var in df_ej["variable"].unique():
    sub = df_ej[df_ej["variable"]==var]
    ax.plot(sub["radius_km"], sub["rho"], marker="o", markersize=7, linewidth=2, label=var)
ax.axhline(0,color="gray",lw=1,ls=":"); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig(f"{FIGDIR}/fig6_ej_correlations.png", dpi=DPI, bbox_inches="tight"); plt.close()

# Fig 7: Pareto frontier
sim = pd.read_csv("/kaggle/working/mitigation_simulation_results_v2.csv")
strategy_colors = {"S1_emissions_only":"#B0B0B0","S2_persistence_aware":"#5DA5DA","S3_accountability_aware":"#FAA43A","S4_justice_aware":"#D62728"}
fig, ax = plt.subplots(figsize=(7.5,6))
for strat in sim["strategy"].unique():
    sub = sim[sim["strategy"]==strat].sort_values("pct_emissions_captured")
    ax.plot(sub["pct_emissions_captured"], sub["pct_vulnerability_benefit"], marker="o", markersize=8, linewidth=2.2,
            color=strategy_colors[strat], label=strat)
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig(f"{FIGDIR}/fig7_pareto_frontier.png", dpi=DPI, bbox_inches="tight"); plt.close()

print("All 7 figures saved to", FIGDIR)

All 7 figures saved to /kaggle/working/figures


In [18]:
# ============================================================
# FINAL PACKAGING — zip everything for the repository
# ============================================================
import shutil
PKG_DIR = "/kaggle/working/final_package"
os.makedirs(PKG_DIR, exist_ok=True)
for f in os.listdir("/kaggle/working"):
    full = f"/kaggle/working/{f}"
    if os.path.isfile(full) and (f.endswith(".parquet") or f.endswith(".csv") or f.endswith(".pkl")):
        shutil.copy2(full, PKG_DIR)
shutil.copytree(FIGDIR, f"{PKG_DIR}/figures", dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/methane_paper_final", "zip", PKG_DIR)
print("Packaged: /kaggle/working/methane_paper_final.zip")

Packaged: /kaggle/working/methane_paper_final.zip
